# Creating Custom Equity Indexes

**FIN 684 - Investment Theory / Advanced Corporate Finance (Prof. Sury), Project I**  
Team "Lehman Brothers 2.0": Nathan Arimilli, Joseph Bailey, Shahmir Javed

This notebook builds three custom U.S. equity indexes (equal-, value-, and price-weighted) from
CRSP monthly data via WRDS, then benchmarks them against the SPY, DIA, and QQQ ETFs.

**Pipeline**
1. Pull the CRSP monthly panel (U.S. common shares on NYSE/AMEX/NASDAQ) and compute market cap.
2. Merge delisting returns to compute effective monthly returns (mitigates survivorship bias).
3. Build Top-N indexes with monthly reconstitution using only lagged (t-1) information (no look-ahead).
4. Pull ETF benchmark series from CRSP.
5. Compute the correlation matrix of monthly log returns and plot results.

> **Reproducibility note.** Running this notebook requires a WRDS account with a CRSP subscription.
> CRSP data is licensed and cannot be redistributed, so no raw data is included in this repo.
> The headline results in the report were produced with the default configuration (Top-100, 2000-2024).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wrds
from typing import Dict, List, Optional
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Configuration

Set the analysis parameters here to run end-to-end without interactive prompts.
Set `INTERACTIVE = True` to be prompted for the date range and number of stocks instead.
WRDS credentials are read from your `~/.pgpass` file (set up once via `wrds.Connection()`),
so no passwords are stored in this notebook.

In [ ]:
# --- Analysis parameters (the example specification used in the report) ---
START_YEAR  = 2000          # 2000-2024
END_YEAR    = 2024          # 2000-2024
TOP_N       = 100           # number of stocks in the index (1-2000)
ETF_TICKERS = ['SPY', 'DIA', 'QQQ']

INTERACTIVE = False         # True -> prompt for start/end year and Top-N
SAVE_FIGURE = True          # save the results figure to ../output/
FIGURE_PATH = '../output/index_comparison.png'

START_DATE = f'{START_YEAR}-01-01'
END_DATE   = f'{END_YEAR}-12-31'

## 1. Connect to WRDS

In [ ]:
def connect_wrds() -> wrds.Connection:
    """Open a WRDS connection (credentials come from ~/.pgpass)."""
    print("Connecting to WRDS...")
    db = wrds.Connection()
    print("Connected.")
    return db

## 2. CRSP monthly panel

Pull price, shares outstanding, and monthly return from the CRSP Monthly Stock File (MSF),
joined to the name history (`msenames`) on a time-bounded key so each month gets the correct
share/exchange codes. We keep only U.S. common shares (`shrcd` 10, 11) on NYSE/AMEX/NASDAQ
(`exchcd` 1, 2, 3). Prices are taken as absolute values (CRSP stores a negative sign for
bid/ask-average quotes), shares outstanding are scaled from thousands, and market cap is
price x shares. Data is pulled year-by-year to keep memory in check.

In [ ]:
def get_crsp_monthly_panel(db: wrds.Connection, start: str, end: str) -> pd.DataFrame:
    """Pull the CRSP monthly panel with security descriptors via a time-bounded join."""
    print(f"Querying CRSP data from {start} to {end}...")
    years = range(pd.to_datetime(start).year, pd.to_datetime(end).year + 1)
    frames: List[pd.DataFrame] = []

    for y in tqdm(years, desc="CRSP query"):
        chunk_start = f"{y}-01-01" if y > pd.to_datetime(start).year else start
        chunk_end   = f"{y}-12-31" if y < pd.to_datetime(end).year else end

        q = f"""
        SELECT m.date, m.permno, m.prc, m.shrout, m.ret,
               n.shrcd, n.exchcd
        FROM crsp.msf AS m
        JOIN crsp.msenames AS n
          ON m.permno = n.permno
         AND m.date BETWEEN n.namedt AND COALESCE(n.nameendt, DATE '9999-12-31')
        WHERE m.date BETWEEN '{chunk_start}' AND '{chunk_end}'
          AND n.shrcd IN (10, 11)     -- U.S. common shares
          AND n.exchcd IN (1, 2, 3)   -- NYSE, AMEX, NASDAQ
          AND m.prc IS NOT NULL
          AND m.shrout IS NOT NULL
        ORDER BY m.date, m.permno;
        """
        frames.append(db.raw_sql(q))

    df = pd.concat(frames, ignore_index=True)
    df['date']   = pd.to_datetime(df['date'])
    df['prc']    = pd.to_numeric(df['prc'], errors='coerce').abs()        # CRSP sign convention
    df['shrout'] = pd.to_numeric(df['shrout'], errors='coerce') * 1000.0  # thousands -> shares
    df['ret']    = pd.to_numeric(df['ret'], errors='coerce')
    df['mktcap'] = df['prc'] * df['shrout']
    return df

## 3. Delisting returns

To avoid survivorship bias we merge delisting returns from `msedelist` and combine them with the
regular return: `r_eff = (1 + r_CRSP) * (1 + r_delist) - 1`. Missing delisting returns are treated
as 0, a conservative and standard assumption.

In [ ]:
def get_delistings(db: wrds.Connection, start: str, end: str) -> pd.DataFrame:
    """Get delisting returns to mitigate survivorship bias."""
    print("Querying delisting returns...")
    q = f"""
    SELECT permno, dlstdt as date, dlret
    FROM crsp.msedelist
    WHERE dlstdt BETWEEN '{start}' AND '{end}'
    """
    delist = db.raw_sql(q, date_cols=['date'])
    delist['dlret'] = pd.to_numeric(delist['dlret'], errors='coerce')
    return delist


def add_effective_returns(panel: pd.DataFrame, delist: pd.DataFrame) -> pd.DataFrame:
    """Merge delisting returns and compute effective monthly returns."""
    print("Computing effective returns with delisting adjustments...")
    out = panel.merge(delist, on=['permno', 'date'], how='left')
    out['ret_eff'] = (1.0 + out['ret'].fillna(0.0)) * (1.0 + out['dlret'].fillna(0.0)) - 1.0
    return out

## 4. Index construction

For each month *t* we select the Top-N constituents by market cap **as of t-1** and set the
equal-, value-, and price-weights from t-1 information, then grow the index from t-1 to t using
the effective returns at *t*. Using only lagged information avoids look-ahead bias; months with
incomplete data carry the prior level forward. All three indexes start at a base level of 100.

- **Equal-weighted:** simple average of constituent gross returns.
- **Value-weighted:** weights proportional to t-1 market cap.
- **Price-weighted:** weights proportional to t-1 share price.

In [ ]:
def build_topn_indexes(panel: pd.DataFrame, top_n: int) -> pd.DataFrame:
    """Construct Equal-, Value-, and Price-Weighted index levels (base = 100)."""
    print(f"Building Top-{top_n} indexes with monthly reconstitution...")
    panel = panel.sort_values(['date', 'permno'])
    months = sorted(panel['date'].unique())
    by_month = {d: g for d, g in panel.groupby('date')}

    ew_level = [100.0]
    vw_level = [100.0]
    pw_level = [100.0]

    for i in tqdm(range(1, len(months)), desc="Index build"):
        t_1, t = months[i - 1], months[i]
        g_t1 = by_month[t_1].dropna(subset=['mktcap', 'prc'])  # constituents/weights from t-1
        g_t  = by_month[t]

        if g_t1.empty or g_t.empty:                            # carry forward on missing data
            ew_level.append(ew_level[-1]); vw_level.append(vw_level[-1]); pw_level.append(pw_level[-1])
            continue

        chosen = g_t1.sort_values('mktcap', ascending=False).head(top_n).set_index('permno')
        g_t = g_t.set_index('permno')
        common = chosen.index.intersection(g_t.index)

        if len(common) == 0:
            ew_level.append(ew_level[-1]); vw_level.append(vw_level[-1]); pw_level.append(pw_level[-1])
            continue

        r_t = g_t.loc[common, 'ret_eff'].astype(float)

        # Equal-weighted
        r_ew = r_t.dropna()
        ew_gross = (1.0 + r_ew).mean() if not r_ew.empty else 1.0

        # Value-weighted (weights from t-1 market cap)
        w_vw = chosen.loc[common, 'mktcap'].astype(float)
        mask = r_t.notna()
        w_vw, r_vw = w_vw.loc[mask], r_t.loc[mask]
        vw_gross = ((1.0 + r_vw) * (w_vw / w_vw.sum())).sum() if (not w_vw.empty and w_vw.sum() > 0) else 1.0

        # Price-weighted (weights from t-1 price)
        w_pw = chosen.loc[common, 'prc'].astype(float)
        w_pw, r_pw = w_pw.loc[mask], r_t.loc[mask]
        pw_gross = ((1.0 + r_pw) * (w_pw / w_pw.sum())).sum() if (not w_pw.empty and w_pw.sum() > 0) else 1.0

        ew_level.append(ew_level[-1] * float(ew_gross))
        vw_level.append(vw_level[-1] * float(vw_gross))
        pw_level.append(pw_level[-1] * float(pw_gross))

    idx_months = pd.to_datetime(months[1:])
    ew = pd.Series(ew_level[1:], index=idx_months, name='Equal_Weight')
    vw = pd.Series(vw_level[1:], index=idx_months, name='Value_Weight')
    pw = pd.Series(pw_level[1:], index=idx_months, name='Price_Weight')
    return pd.concat([ew, vw, pw], axis=1)

## 5. ETF benchmarks

We map each ETF ticker to its CRSP `permno` via `stocknames`, then compound its monthly returns
into a base-100 index level. SPY, DIA, and QQQ all have full history back to 2000. (We originally
considered IWM and VTI but they lack complete coverage from 2000, so DIA stands in as the third
benchmark alongside SPY and QQQ.)

In [ ]:
def get_etf_index(db: wrds.Connection, ticker: str, start: str, end: str) -> Optional[pd.Series]:
    """Build a 100-based index level for an ETF by compounding monthly returns."""
    print(f"Getting {ticker} data...")
    permno_query = f"""
    SELECT DISTINCT permno, namedt, nameenddt
    FROM crsp.stocknames
    WHERE ticker = '{ticker}'
      AND namedt <= '{end}'
      AND (nameenddt >= '{start}' OR nameenddt IS NULL)
    ORDER BY namedt DESC
    """
    permnos = db.raw_sql(permno_query, date_cols=['namedt', 'nameenddt'])
    if permnos.empty:
        print(f"  Warning: no PERMNO found for {ticker}")
        return None

    permno = permnos.iloc[0]['permno']
    q = f"""
    SELECT date, ret
    FROM crsp.msf
    WHERE permno = {permno}
      AND date BETWEEN '{start}' AND '{end}'
    ORDER BY date;
    """
    d = db.raw_sql(q, date_cols=['date'])
    if d.empty:
        print(f"  Warning: no data found for {ticker}")
        return None

    d = d.sort_values('date')
    d['ret'] = pd.to_numeric(d['ret'], errors='coerce').fillna(0.0)
    idx = (1.0 + d['ret']).cumprod() * 100.0
    idx.index = d['date']
    idx.name = ticker
    return idx

## 6. Correlation of monthly log returns

Align all series on their common dates and correlate monthly log returns
(`ln(level_t / level_{t-1})`).

In [ ]:
def align_and_corr(levels: List[pd.Series]) -> pd.DataFrame:
    """Align series on common dates and compute the correlation of monthly log returns."""
    if not levels:
        return pd.DataFrame()
    common = set(levels[0].index)
    for s in levels[1:]:
        common &= set(s.index)
    common = sorted(common)
    if len(common) < 2:
        print("Warning: not enough common dates for correlation")
        return pd.DataFrame()
    aligned = [s.loc[common] for s in levels]
    panel = pd.concat(aligned, axis=1)
    log_ret = np.log(panel / panel.shift(1)).dropna(how='any')
    return log_ret.corr()

## 7. Plots

In [ ]:
def plot_indexes(custom_df: pd.DataFrame, etf_map: Dict[str, pd.Series],
                 corr_mat: pd.DataFrame, top_n: int, save_path: Optional[str] = None):
    """Plot custom indexes vs ETF benchmarks and the correlation heatmap."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    for col in custom_df.columns:
        ax1.plot(custom_df.index, custom_df[col], label=col, linewidth=2)
    for name, series in etf_map.items():
        aligned = series.reindex(custom_df.index).dropna()
        if not aligned.empty:
            ax1.plot(aligned.index, aligned.values, label=name, linestyle='--', alpha=0.7)
    ax1.set_title(f'Top-{top_n} Custom Indexes vs ETF Benchmarks')
    ax1.set_ylabel('Index Level (Base = 100)')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    if not corr_mat.empty:
        sns.heatmap(corr_mat, annot=True, cmap="coolwarm", ax=ax2, fmt='.2f')
        ax2.set_title('Correlation Matrix (Monthly Log Returns)')

    plt.tight_layout()
    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    plt.show()

## 8. Run the analysis

In [ ]:
def get_user_inputs():
    """Prompt for start year, end year, and Top-N (used when INTERACTIVE = True)."""
    while True:
        try:
            sy = int(input("Start year (2000-2024): "))
            ey = int(input("End year (2000-2024): "))
            if 2000 <= sy <= 2024 and 2000 <= ey <= 2024 and sy <= ey:
                break
            print("Invalid years.")
        except ValueError:
            print("Please enter valid years.")
    while True:
        try:
            n = int(input("Number of stocks (1-2000): "))
            if 1 <= n <= 2000:
                break
            print("Must be between 1 and 2000.")
        except ValueError:
            print("Please enter a number.")
    return f"{sy}-01-01", f"{ey}-12-31", n


def run_complete_analysis(db, start_date, end_date, top_n, save_path=None):
    panel = get_crsp_monthly_panel(db, start_date, end_date)
    print(f"Rows pulled: {len(panel):,}")

    delist = get_delistings(db, start_date, end_date)
    panel = add_effective_returns(panel, delist)

    custom = build_topn_indexes(panel, top_n)

    etf_series: Dict[str, pd.Series] = {}
    for t in ETF_TICKERS:
        s = get_etf_index(db, t, start_date, end_date)
        if s is not None:
            etf_series[t] = s

    all_series = [custom[c] for c in custom.columns] + list(etf_series.values())
    corr_mat = align_and_corr(all_series)

    # --- Total and annualized returns ---
    total = (custom.iloc[-1] / custom.iloc[0] - 1) * 100
    years = (custom.index[-1] - custom.index[0]).days / 365.25
    annual = ((custom.iloc[-1] / custom.iloc[0]) ** (1 / years) - 1) * 100

    print("\nCustom index total (annualized) returns:")
    for name in custom.columns:
        print(f"  {name}: {total[name]:.1f}%  ({annual[name]:.1f}%/yr)")

    print("\nETF total returns:")
    for t, s in etf_series.items():
        a = s.reindex(custom.index).dropna()
        if not a.empty:
            print(f"  {t}: {(a.iloc[-1] / a.iloc[0] - 1) * 100:.1f}%")

    print("\nCorrelation matrix (monthly log returns):")
    print(corr_mat.round(2).to_string())

    plot_indexes(custom, etf_series, corr_mat, top_n, save_path=save_path)
    return custom, etf_series, corr_mat


db = connect_wrds()
try:
    if INTERACTIVE:
        sd, ed, n = get_user_inputs()
    else:
        sd, ed, n = START_DATE, END_DATE, TOP_N
    custom, etfs, corr = run_complete_analysis(
        db, sd, ed, n, save_path=FIGURE_PATH if SAVE_FIGURE else None
    )
finally:
    db.close()
    print("WRDS connection closed.")